# EV Route Optimizer — Keşifsel Veri Analizi (EDA)

Bu notebook, Faz 4 kapsamında sentetik sürüş verisini inceler ve LightGBM modelinin formül tabanlı (heuristic) baseline'a karşı performansını analiz eder.

**Akış:**
1. Veri yükleme ve sağlık kontrolleri
2. Hedef değişken (`target_energy_kwh`) dağılımı
3. Sürüş koşullarının (hız, sıcaklık, eğim, rota tipi) tüketime etkisi
4. Araç bazında tüketim profili
5. LightGBM modeli: baseline vs ML karşılaştırması
6. Özellik önemi (feature importance) ve çıkarımlar

## 1. Ortam hazırlığı

In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

# Proje kök dizinini path'e ekle, notebook 'notebooks/' içinden çalıştırılıyor.
ROOT_DIR = Path.cwd().resolve()
if ROOT_DIR.name == "notebooks":
    ROOT_DIR = ROOT_DIR.parent
sys.path.insert(0, str(ROOT_DIR))

DATA_PATH = ROOT_DIR / "app" / "data" / "synthetic_drive_data.csv"
VEHICLES_PATH = ROOT_DIR / "app" / "data" / "vehicles.json"
MODEL_PATH = ROOT_DIR / "ml" / "models" / "lgbm_v1.joblib"

print("Proje kökü    :", ROOT_DIR)
print("Veri          :", DATA_PATH, "(var mı?", DATA_PATH.exists(), ")")
print("Araç DB       :", VEHICLES_PATH, "(var mı?", VEHICLES_PATH.exists(), ")")
print("Model artefakt:", MODEL_PATH, "(var mı?", MODEL_PATH.exists(), ")")

In [ ]:
import json
import warnings

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 200)

try:
    import matplotlib.pyplot as plt
    MPL_OK = True
except ImportError:
    MPL_OK = False
    print("matplotlib kurulu değil; grafikler atlanacak.")

## 2. Veriyi yükle ve tanı

5000 satırlık sentetik sürüş veri seti: her satır tek bir rota segmentinin enerji tüketimini temsil eder.

In [ ]:
df = pd.read_csv(DATA_PATH)
print("Boyut:", df.shape)
df.head()

In [ ]:
# Temel istatistikler
numeric_cols = [
    "distance_km", "speed_kmh", "grade_pct", "temp_c",
    "start_soc_pct", "target_wh_km", "target_energy_kwh",
]
df[numeric_cols].describe().round(2)

In [ ]:
# Null / NaN kontrolü
nulls = df.isna().sum()
print("NaN içeren kolon sayısı:", int((nulls > 0).sum()))
nulls[nulls > 0]

## 3. Hedef değişken dağılımı

`target_energy_kwh` (segment başına tüketilen enerji) ve `target_wh_km` (birim uzaklık başına tüketim) dağılımları.

In [ ]:
print("target_energy_kwh özet:")
print(df["target_energy_kwh"].describe().round(3))
print("\ntarget_wh_km özet (Wh/km):")
print(df["target_wh_km"].describe().round(2))

In [ ]:
if MPL_OK:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].hist(df["target_energy_kwh"], bins=50, color="#4e79a7")
    axes[0].set_title("target_energy_kwh dağılımı")
    axes[0].set_xlabel("kWh / segment")
    axes[0].set_ylabel("frekans")

    axes[1].hist(df["target_wh_km"], bins=50, color="#f28e2b")
    axes[1].set_title("target_wh_km dağılımı")
    axes[1].set_xlabel("Wh / km")
    plt.tight_layout()
    plt.show()

## 4. Sürüş koşullarının tüketime etkisi

Hız, sıcaklık ve eğimin `target_wh_km` üzerindeki etkisini gözlemle.

In [ ]:
# Korelasyon (sayısal özellikler vs hedef)
corr_cols = ["distance_km", "speed_kmh", "grade_pct", "temp_c",
             "hvac_override_kw", "start_soc_pct", "target_wh_km"]
df[corr_cols].corr()[["target_wh_km"]].round(3).sort_values("target_wh_km", ascending=False)

In [ ]:
if MPL_OK:
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))

    axes[0].scatter(df["speed_kmh"], df["target_wh_km"], alpha=0.15, s=10)
    axes[0].set_title("Hız vs Tüketim")
    axes[0].set_xlabel("Hız (km/h)")
    axes[0].set_ylabel("Wh/km")

    axes[1].scatter(df["temp_c"], df["target_wh_km"], alpha=0.15, s=10, color="#e15759")
    axes[1].set_title("Sıcaklık vs Tüketim")
    axes[1].set_xlabel("Sıcaklık (°C)")

    axes[2].scatter(df["grade_pct"], df["target_wh_km"], alpha=0.15, s=10, color="#59a14f")
    axes[2].set_title("Eğim vs Tüketim")
    axes[2].set_xlabel("Eğim (%)")
    plt.tight_layout()
    plt.show()

In [ ]:
# Rota tipi x terrain etkisi
pivot = df.groupby(["route_type", "terrain_type"])["target_wh_km"].mean().unstack().round(1)
pivot

## 5. Araç bazında tüketim

Hangi araçlar daha verimli, hangileri daha ağır tüketiyor?

In [ ]:
by_vehicle = (
    df.groupby(["vehicle_id", "make", "model"])
      .agg(
          rows=("target_wh_km", "size"),
          mean_wh_km=("target_wh_km", "mean"),
          median_wh_km=("target_wh_km", "median"),
          std_wh_km=("target_wh_km", "std"),
      )
      .reset_index()
      .sort_values("mean_wh_km")
      .round(2)
)
by_vehicle

## 6. LightGBM modeli: eğit, baseline ile kıyasla

`ml.train_model` modülü modeli eğitip joblib olarak kaydeder; `ml.evaluate` ise formül bazlı baseline ile metrik karşılaştırması üretir.

In [ ]:
from ml.train_model import train_model
from ml.evaluate import evaluate_model

artifacts = train_model(
    data_path=DATA_PATH,
    vehicles_path=VEHICLES_PATH,
    model_output_path=MODEL_PATH,
)

report = artifacts.report
print("Eğitim tamam.")
print("Train rows:", report["train_rows"], "| Test rows:", report["test_rows"])
print("Latency/istek (ms):", report["latency_ms_per_request"])

In [ ]:
comparison = pd.DataFrame(
    {
        "baseline": report["baseline_metrics"],
        "lightgbm": report["model_metrics"],
    }
).round(4)
print("Baseline (formül) vs LightGBM:")
comparison

In [ ]:
# Değerlendirme raporunu ml/reports/eval_v1.json olarak sakla
report_path = ROOT_DIR / "ml" / "reports" / "eval_v1.json"
eval_report = evaluate_model(
    data_path=DATA_PATH,
    vehicles_path=VEHICLES_PATH,
    model_path=MODEL_PATH,
    save_report_path=report_path,
)

print("Rapor yolu:", report_path)
print("Model baseline'dan iyi mi?:", eval_report["model_better_than_baseline"])
print("İyileşme yüzdeleri    :", eval_report["improvements"])

## 7. Özellik önemi (Feature Importance)

In [ ]:
pipeline = artifacts.model_pipeline
lgbm_model = pipeline.named_steps["model"]
preprocessor = pipeline.named_steps["preprocessor"]

try:
    feature_names = list(preprocessor.get_feature_names_out())
except Exception:
    feature_names = [f"f_{i}" for i in range(len(lgbm_model.feature_importances_))]

importances = pd.DataFrame(
    {
        "feature": feature_names,
        "importance": lgbm_model.feature_importances_,
    }
).sort_values("importance", ascending=False)

importances.head(15)

In [ ]:
if MPL_OK:
    top = importances.head(12).iloc[::-1]
    fig, ax = plt.subplots(figsize=(9, 5))
    ax.barh(top["feature"], top["importance"], color="#76b7b2")
    ax.set_title("LightGBM — Özellik Önemi (ilk 12)")
    ax.set_xlabel("importance")
    plt.tight_layout()
    plt.show()

## 8. Çıkarımlar

- **Baseline (formül)** ile **LightGBM** arasındaki fark `ml/reports/eval_v1.json` dosyasında kayıtlı.
- Model sentetik veriyle eğitildiği için gerçek bir tahminciden ziyade formülün pürüzsüzleştirilmiş bir proxy'sidir. MVP'de bu, **iyileştirme katmanı** olarak konumlanmıştır; çekirdek sistem ML olmadan da ayakta kalır (`ModelService` fallback).
- Gelecek adımlar:
  - **Quantile LightGBM** (p10/p50/p90) ile risk farkındalıklı enerji tahmini.
  - **Monotonic constraints** (mesafe↑ ⇒ enerji↑) ekleyerek domain garantisi.
  - **SHAP** ile segment bazlı açıklanabilirlik.
  - Gerçek telemetri verisiyle yeniden eğitim (v2).